In [ ]:
import pyxdf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib widget


fname_xdf = "data.xdf"
fname_csv = "data.csv"
fname_marker_csv = "marker.csv"

# To start LabRecorder with the config file:
# open -a "/Applications/LabRecorder/LabRecorder.app" --args -c /Applications/LabRecorder/LabRecorder.cfg

In [ ]:
# read the xdf file

data, header = pyxdf.load_xdf(fname_xdf)

# list the streams
for stream in data:
    print(f"Stream Name: {stream['info']['name'][0]}")
    print(f"Stream Type: {stream['info']['type'][0]}")
    print(f"Number of Samples: {len(stream['time_series'])}")
    print()

In [ ]:
# print all info fields

import pprint

for stream in data:
    print("=" * 40)
    print(f"Complete Info for {stream['info']['name'][0]}:")
    pprint.pprint(stream["info"], indent=2, width=120)

In [ ]:
# print all markers from the markers streams in the xdf file
marker_streams = [s for s in data if s["info"]["type"][0] == "Markers"]

mouse_markers = [s for s in marker_streams if s["info"]["name"][0] == "MouseMarkers"]


print(f"Markers from XDF: (n={len(mouse_markers[0]['time_stamps'])})")
for sample, timestamp in zip(
    mouse_markers[0]["time_series"], mouse_markers[0]["time_stamps"]
):
    text = sample[0]
    time_ms = int(timestamp * 1000.0)
    print(f"  {time_ms}: {text}")
print()

# print all markers from the marker.csv file (skip header)
marker_df = pd.read_csv(fname_marker_csv, skiprows=3)
n_rows = marker_df.shape[0]
print(f"Markers from CSV: (n={n_rows})")
for index, row in marker_df.iterrows():
    print(f"  {row['milliseconds']:.0f}: {row['marker']}")

In [ ]:
# print all data from the data stream in the xdf file
data_streams = [s for s in data if s["info"]["type"][0] == "MoCap"]
mouse_data = [s for s in data_streams if s["info"]["name"][0] == "MouseData"][0]

print("")
print(f"Data from XDF: (n={len(mouse_data['time_stamps'])})")
for sample, timestamp in zip(mouse_data["time_series"], mouse_data["time_stamps"]):
    time_ms = int(timestamp * 1000.0)
    # print all values in the sample
    print(f"  {time_ms}: ", end="")
    for i, value in enumerate(sample):
        print(f"{value}", end=", " if i < len(sample) - 1 else "")
print()
# print all data from the data.csv file (skip header)
data_df = pd.read_csv(fname_csv, skiprows=3)
# print the column names
print(data_df.columns)

print("Data from CSV: (n={})".format(data_df.shape[0]))
for index, row in data_df.iterrows():
    # print all values in the row
    for col in data_df.columns:
        print(f"  {col}: {row[col]}", end=", ")
    print()
print()

In [ ]:
# plot the difference between the timestamp columns in the data.csv file

# add mouse_data["time_stamps"] to data_df as a new column "timestamp" in milliseconds
data_df["timestamp"] = [int(ts * 1000.0) for ts in mouse_data["time_stamps"]]

# print the columns names
print(data_df.columns)


data_df["timestamp_diff"] = data_df["timestamp"] - data_df["event_timestamp"]
data_df["timestamp_delta"] = data_df["timestamp"] - data_df["unix_timestamp"]


# make a boxplot of the timestamp differences
# there is a constant offset between the LSL timestamp and the event/unix timestamps,
# but we want to see the variability around this offset
# hence we subtract the median from each difference to center the boxplot around zero
plt.figure()
event_timestamp_delta = data_df["timestamp_diff"] - data_df["timestamp_diff"].median()
unix_timestamp_delta = data_df["timestamp_delta"] - data_df["timestamp_delta"].median()
plt.boxplot(
    [event_timestamp_delta, unix_timestamp_delta],
    tick_labels=[
        f"lsl timestamp - event_timestamp \n{data_df['timestamp_diff'].median()}",
        f"lsl timestamp - unix_timestamp \n{data_df['timestamp_delta'].median()}",
    ],
)
plt.title("Boxplot of timestamp differences")
plt.ylabel("Difference (ms)")
plt.show()

In [ ]:
# assert that the data values from LSL and CSV are equal within a tolerance

data_values_lsl = np.array(mouse_data["time_series"])
n_cols_lsl = data_values_lsl.shape[1]
n_rows_lsl = data_values_lsl.shape[0]

data_df = pd.read_csv(fname_csv, skiprows=3)
data_values_csv = data_df.to_numpy()
n_cols_csv = data_values_csv.shape[1]
n_rows_csv = data_values_csv.shape[0]

assert (
    n_cols_lsl == n_cols_csv
), f"Number of columns mismatch: LSL={n_cols_lsl}, CSV={n_cols_csv}"
assert (
    n_rows_lsl == n_rows_csv
), f"Number of rows mismatch: LSL={n_rows_lsl}, CSV={n_rows_csv}"

for i in range(n_rows_lsl):
    for j in range(n_cols_lsl):
        val_lsl = data_values_lsl[i, j]
        val_csv = data_values_csv[i, j]
        assert (
            val_lsl == val_csv
        ), f"Data mismatch at row {i}, column {j}: LSL={val_lsl}, CSV={val_csv}"

In [ ]:
# get the max difference between LSL and CSV data values for each column
max_diffs = []
for j in range(n_cols_lsl):
    col_diffs = []
    for i in range(n_rows_lsl):
        val_lsl = data_values_lsl[i, j]
        val_csv = data_values_csv[i, j]
        diff = abs(val_lsl - val_csv)
        col_diffs.append(diff)
    max_diff = max(col_diffs)
    max_diffs.append(max_diff)

print("Max differences between LSL and CSV data values for each column:")
for j in range(n_cols_lsl):
    print(f"  Column {j}: {max_diffs[j]:.20f}")